In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"
DEVICE = "cpu"

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from pathlib import Path

from open_vocab_mot.data import (
    DukeMTMCVideoDataset, 
    DukeSplit,
    DukeMTMCVideoDatasetVideoKPFBatchSampler,
    collate_duke_mtmc_video_ds
)
from open_vocab_mot.definitions import DUKEMTMC_VIDEO_REID_PATH, DUKEMTMC_VIDEO_REID_SIDECAR_PATH
from aidan_lib.models.dino_lib_compiled import DINOv3CompiledHarness
from open_vocab_mot.models import HierarchicalVideoReIDTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
# Note: Update this checkpoint path if you have trained weights you want to use
checkpoint_path = "../../weights/reid_transformer_duke_parallel.pt" 

dino_harness = DINOv3CompiledHarness(
    checkpoint="facebook/dinov3-vitl16-pretrain-lvd1689m",
    device=device,
    dtype=torch.bfloat16,
    max_side_len=1024,
    warmup=False
)

model = HierarchicalVideoReIDTransformer(
    input_dim=dino_harness.embedding_dim,
    frame_transformer_dim=512,
    frame_contrastive_dim=256,
    frame_num_heads=8,
    frame_num_layers=4,
    video_transformer_dim=384,
    video_contrastive_dim=256,
    video_num_heads=6,
    video_num_layers=3,
    dropout=0.1
).to(device)

if Path(checkpoint_path).exists():
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded weights from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint {checkpoint_path} not found. Using untrained model.")

model.eval()

# Load dataset (using TRAIN split, but you can change to QUERY/GALLERY)
ds = DukeMTMCVideoDataset(
    ds_root=DUKEMTMC_VIDEO_REID_PATH,
    main_split=DukeSplit.TRAIN,
    sidecar_root=DUKEMTMC_VIDEO_REID_SIDECAR_PATH,
    load_image_pil=True,  # Needed for visualization
    load_image_tensor=True,
    load_segmentations=True
)
print(f"Loaded dataset with {len(ds)} total frames")


In [ ]:
attention_weights = {
    "frame": [],
    "video": []
}

def get_attention_hook(name):
    def hook(module, input, output):
        # MultiheadAttention output is a tuple: (attn_output, attn_output_weights)
        attention_weights[name].append(output[1].detach().cpu())
    return hook

# Hook into the last layer of each transformer
model.frame_transformer.layers[-1].self_attn.register_forward_hook(get_attention_hook("frame"))
model.video_transformer.layers[-1].self_attn.register_forward_hook(get_attention_hook("video"))
print("Registered forward hooks.")


In [ ]:
from open_vocab_mot.data import DukeMTMCItemBatch
num_frames_to_visualize = 16

sampler = DukeMTMCVideoDatasetVideoKPFBatchSampler(
    ds,
    batches_per_epoch=1,
    num_people_per_batch=1,
    num_views_per_person=1,
    num_frames_per_view=num_frames_to_visualize,
    allow_same_person_same_view=False,
    allow_reduced_views_per_person=False,
    allow_resampling_sample_indices=True, # Allow resampling if the view is short
)

loader = DataLoader(
    dataset=ds,
    collate_fn=collate_duke_mtmc_video_ds,
    batch_sampler=sampler,
)

# Get the first (and only) batch
batch: DukeMTMCItemBatch = next(iter(loader))

frames = batch.frame_tensors
pil_frames = batch.frames
segmentations = batch.segmentations

print(f"Extracted a video sequence with {len(frames)} frames.")
print(f"Person ID: {batch.person_ids[0]}, Camera ID: {batch.camera_ids[0]}")

print("Extracting DINO embeddings...")
dino_embeddings_batch = dino_harness.match_bool_segmentations_to_dino(
    [f.to(device) for f in frames], 
    segmentations
)

# Structure embeddings for the ReID model
video_embeddings = []
for frame_dino in dino_embeddings_batch:
    if len(frame_dino) > 0:
        video_embeddings.append(frame_dino[0].dino_embeddings)

# Wrap in a list because the model expects a batch of videos
video_embeddings_input = [video_embeddings]

# Clear hooks before forward pass
attention_weights["frame"].clear()
attention_weights["video"].clear()

import torch.nn as nn

# 1. Save the original forward method
original_forward = nn.MultiheadAttention.forward

# 2. Define our patched forward that forces need_weights=True
def forward_with_weights(*args, **kwargs):
    kwargs['need_weights'] = True
    return original_forward(*args, **kwargs)

# 3. Apply the patch
nn.MultiheadAttention.forward = forward_with_weights

# 4. Run your forward pass
with torch.no_grad():
    output = model(video_embeddings_input)

# 5. Restore the original behavior
nn.MultiheadAttention.forward = original_forward

print("Successfully ran forward pass and captured weights.")


print("Successfully ran forward pass.")


In [ ]:
# Extract the attention map from the Video Transformer
# Shape of vid_attn is (batch_size, tgt_len, src_len)
vid_attn = attention_weights["video"][0][0]

# Get the attention from the [CLS] token (index 0) to all frames (index 1 to N)
cls_to_frame_attn = vid_attn[0, 1:].numpy()

plt.figure(figsize=(10, 4))
bars = plt.bar(range(len(cls_to_frame_attn)), cls_to_frame_attn, color='skyblue')
plt.title("Attention from Video [CLS] to Each Frame")
plt.xlabel("Frame Index")
plt.ylabel("Attention Weight")
plt.xticks(range(len(cls_to_frame_attn)))

# Highlight the frame with the highest attention
best_frame_idx = np.argmax(cls_to_frame_attn)
bars[best_frame_idx].set_color('salmon')
plt.show()

print(f"Frame {best_frame_idx} has the highest attention weight.")


In [ ]:
frame_index = 12

# Extract the attention map from the Frame Transformer
# Shape is (total_frames_in_batch, tgt_len, src_len)
frame_attn = attention_weights["frame"][0] 

# Get attention from Frame CLS (index 0) to its patches (index 1 onwards)
cls_to_patch_attn = frame_attn[frame_index, 0, 1:].numpy()

# Get the original image and corresponding DINO bounding boxes for this frame
pil_image = pil_frames[frame_index]
dino_seg = dino_embeddings_batch[frame_index][0]
bboxes = dino_seg.dino_bboxes.cpu().numpy() # Shape (num_patches, 4)

# Normalize attention weights for visualization scaling (0 to 1)
if len(cls_to_patch_attn) > len(bboxes):
    cls_to_patch_attn = cls_to_patch_attn[:len(bboxes)]

cls_to_patch_attn = cls_to_patch_attn - cls_to_patch_attn.min()
if cls_to_patch_attn.max() > 0:
    cls_to_patch_attn = cls_to_patch_attn / cls_to_patch_attn.max()

# Plot the image and overlay the patch attention
fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(pil_image)

for i, (px1, py1, px2, py2) in enumerate(bboxes):
    attn = cls_to_patch_attn[i]
    # Draw a rectangle with alpha opacity proportional to the attention weight
    rect = patches.Rectangle(
        (px1, py1), px2 - px1, py2 - py1, 
        linewidth=0, edgecolor='none', facecolor='red', alpha=float(attn * 0.75)
    )
    ax.add_patch(rect)

plt.title(f"Patch-Level Attention Overlay for Frame {frame_index}")
plt.axis('off')
plt.show()
